# Geometry: longitude conventions, bound strings, cell areas, points to zones

See `docs/conventions.md` for the longitude convention and why the CRS metadata never changes.

In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import MultiPolygon, Point, box

from idd_spatial_analysis.constants import EARTH_RADIUS_KM, WGS84, WORLD_CYLINDRICAL_EQUAL_AREA
from idd_spatial_analysis.geometry import (
    cell_area_km2,
    parse_bound,
    points_to_zones,
    split_antimeridian,
    to_lon_180,
    to_lon_360,
)

In [ ]:
# A polygon crossing the prime meridian becomes two pieces at 0 and 360, and comes back whole.
poly = box(-10, -5, 10, 5)
moved = to_lon_360(poly)
print(moved.geom_type, moved.bounds)
back = to_lon_180(moved)
print(back.geom_type, back.bounds, back.symmetric_difference(poly).area)

In [ ]:
# Fiji-style: halves split at the antimeridian by a GIS weld back into one polygon in 0..360.
halves = MultiPolygon([box(170, -20, 180, -10), box(-180, -20, -170, -10)])
whole = to_lon_360(halves)
print(whole.geom_type, whole.bounds)
print([piece.bounds for piece in split_antimeridian(whole)])

In [ ]:
# Bound strings with an explicit wrap.
for text, wrap in (("260E", "none"), ("260E", "180"), ("100W", "360"), ("45S", "none")):
    print(text, wrap, parse_bound(text, wrap=wrap))

In [ ]:
# Exact spherical areas: one-degree rows summed over the globe give the sphere.
areas = cell_area_km2(np.arange(90, -91, -1), 1.0)
print(areas[:3], areas[88:91])
print(areas.sum() * 360, 4 * np.pi * EARTH_RADIUS_KM**2)

In [ ]:
# Points to the zone containing them; <NA> when none does, which with the default "within"
# predicate includes a point on a shared border; with "intersects" the last-listed zone wins.
zones = gpd.GeoDataFrame(
    {"zone": np.array([1, 2], dtype="int32")},
    geometry=[box(0, 0, 10, 10), box(10, 0, 20, 10)],
    crs=WGS84,
)
points = gpd.GeoDataFrame(
    geometry=[Point(5, 5), Point(15, 5), Point(10, 5), Point(50, 50)], crs=WGS84
)
print(points_to_zones(points, zones, "zone"))
print(
    points_to_zones(
        points.to_crs(WORLD_CYLINDRICAL_EQUAL_AREA), zones, "zone", predicate="intersects"
    )
)